💡 **Environment:** `clamp-analyses`  


# Description

Projects S-PrediXcan MASHR gene-trait z-scores (49 tissues) into the GTEx CLAMP latent space.

**Input**: raw S-PrediXcan pkl files from `00_spredixcan_projection_archs4`.

**Output**: `02_spredixcan_projection_gtex/spredixcan/{stem}-projection-gtex.pkl` (LVs × traits per tissue).


# Modules loading


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

from pyprojroot import here


# Settings


In [3]:
MODEL_KEY = 'gtex'
CLAMP_MODEL_FILE = here('output/01_model_building/02_gtex/10_CLAMP_hall/CLAMPfull_hall.rds')
display(CLAMP_MODEL_FILE)
assert CLAMP_MODEL_FILE.exists()


PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/01_model_building/02_gtex/10_CLAMP_hall/CLAMPfull_hall.rds')

In [4]:
# Input: raw S-PrediXcan files from 00_spredixcan_projection_archs4
SPREDIXCAN_RAW_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/00_spredixcan_projection_archs4') / 'spredixcan' / 'raw'
display(SPREDIXCAN_RAW_DIR)
assert SPREDIXCAN_RAW_DIR.exists()

# Output
OUTPUT_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/02_spredixcan_projection_gtex')
SPREDIXCAN_PROJ_DIR = OUTPUT_DIR / 'spredixcan'
SPREDIXCAN_PROJ_DIR.mkdir(parents=True, exist_ok=True)
display(SPREDIXCAN_PROJ_DIR)


PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/00_spredixcan_projection_archs4/spredixcan/raw')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/02_spredixcan_projection_gtex/spredixcan')

# Projection helpers


In [5]:
def prepare_clamp_projector(clamp_model_file):
    CLAMP = importr('CLAMP')
    readRDS = ro.r['readRDS']
    clamp = readRDS(str(clamp_model_file))
    print('CLAMP model loaded')

    gene_symbols = list(ro.r['rownames'](clamp.rx2('Z')))
    lv_names = list(ro.r['colnames'](clamp.rx2('Z')))
    print(f'CLAMP genes: {len(gene_symbols)}, LVs: {len(lv_names)}')

    # Map CLAMP gene symbols (HGNC) to Ensembl IDs used by LINCS and S-PrediXcan.
    clusterProfiler = importr('clusterProfiler')
    bitr_result = clusterProfiler.bitr(
        ro.StrVector(gene_symbols),
        fromType='SYMBOL',
        toType='ENSEMBL',
        OrgDb='org.Hs.eg.db',
    )

    with localconverter(ro.default_converter + pandas2ri.converter):
        mapping_df = ro.conversion.rpy2py(bitr_result)

    print(f'Raw mapping shape: {mapping_df.shape}')
    display(mapping_df.head())

    # Keep only 1:1 unambiguous symbol <-> Ensembl mappings.
    dup_symbols = mapping_df['SYMBOL'].duplicated(keep=False)
    dup_ensembl = mapping_df['ENSEMBL'].duplicated(keep=False)
    mapping_1to1 = mapping_df[~dup_symbols & ~dup_ensembl].set_index('SYMBOL')
    print(f'1:1 mappings: {mapping_1to1.shape[0]} / {len(gene_symbols)} CLAMP genes')

    mapped_symbols = mapping_1to1.index.tolist()
    mapped_ensembl = mapping_1to1['ENSEMBL'].tolist()

    subset_Z = ro.r('function(clamp, genes) { clamp$Z <- as.matrix(clamp$Z[genes, ]); clamp }')
    clamp_sub = subset_Z(clamp, ro.StrVector(mapped_symbols))
    print(f'Subsetted CLAMP Z: {len(mapped_symbols)} genes x {len(lv_names)} LVs')

    return CLAMP, clamp_sub, mapped_ensembl, lv_names


def project_to_clamp(data, CLAMP, clamp_sub, mapped_ensembl, lv_names):
    aligned = data.reindex(mapped_ensembl).fillna(0.0).values

    r_mat = ro.r['matrix'](
        ro.FloatVector(aligned.flatten('F')),
        nrow=aligned.shape[0],
        ncol=aligned.shape[1],
    )

    proj_r = CLAMP.projectCLAMP(clamp_sub, newdata=r_mat)

    with localconverter(ro.default_converter + pandas2ri.converter):
        proj_values = ro.conversion.rpy2py(proj_r)

    return pd.DataFrame(proj_values, index=lv_names, columns=data.columns)


# Prepare CLAMP projector


In [6]:
CLAMP, clamp_sub, mapped_ensembl, lv_names = prepare_clamp_projector(CLAMP_MODEL_FILE)


CLAMP model loaded
CLAMP genes: 21613, LVs: 578


R callback write-console: 
  
R callback write-console: 'select()' returned 1:many mapping between keys and columns
  


Raw mapping shape: (18525, 2)


,SYMBOL,ENSEMBL
6,MTND1P23,ENSG00000225972
7,MTND2P28,ENSG00000225630
8,MTCO1P12,ENSG00000237973
9,MTCO2P12,ENSG00000229344
10,MTATP8P1,ENSG00000240409


R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: In (function (geneID, fromType, toType, OrgDb, drop = TRUE)  :  
R callback write-console: 
   
R callback write-console:  23.79% of input gene IDs are fail to map...
  


1:1 mappings: 15361 / 21613 CLAMP genes
Subsetted CLAMP Z: 15361 genes x 578 LVs


# Project S-PrediXcan tissues


In [7]:
spredixcan_raw_file_list = sorted(
    f for f in SPREDIXCAN_RAW_DIR.glob('*.pkl') if f.name.startswith('spredixcan-')
)
display(len(spredixcan_raw_file_list))
assert len(spredixcan_raw_file_list) == 49

for input_file in spredixcan_raw_file_list:
    base_stem = input_file.stem.removesuffix('-data')
    output_proj = SPREDIXCAN_PROJ_DIR / f'{base_stem}-projection-{MODEL_KEY}.pkl'

    print(input_file.name)
    data = pd.read_pickle(input_file)
    print(f'  shape: {data.shape}')
    assert data.index.is_unique
    assert data.columns.is_unique
    assert not data.isna().any().any()

    print('  projecting through CLAMP...')
    projection = project_to_clamp(
        data, CLAMP, clamp_sub, mapped_ensembl, lv_names
    )
    print(f'    projection shape: {projection.shape}')
    assert not projection.isna().any().any()

    print(f'    saving projection to: {output_proj}')
    projection.to_pickle(output_proj)
    print('')


49

spredixcan-mashr-zscores-Adipose_Subcutaneous-data.pkl
  shape: (14059, 4091)
  projecting through CLAMP...
    projection shape: (578, 4091)
    saving projection to: /home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/02_spredixcan_projection_gtex/spredixcan/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-gtex.pkl

spredixcan-mashr-zscores-Adipose_Visceral_Omentum-data.pkl
  shape: (13938, 4091)
  projecting through CLAMP...
    projection shape: (578, 4091)
    saving projection to: /home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/02_spredixcan_projection_gtex/spredixcan/spredixcan-mashr-zscores-Adipose_Visceral_Omentum-projection-gtex.pkl

spredixcan-mashr-zscores-Adrenal_Gland-data.pkl
  shape: (12894, 4091)
  projecting through CLAMP...
    projection shape: (578, 4091)
    saving projection to: /home/miltondp/projects/clamp/clamp-analyses/output/03_model_bi